In [1]:
import numpy as np
import pandas as pd

In [2]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder

In [3]:
df = pd.read_csv('covid_toy.csv')


df.head()

,age,gender,fever,cough,city,has_covid
0,60,Male,103.0,Mild,Kolkata,No
1,27,Male,100.0,Mild,Delhi,Yes
2,42,Male,101.0,Mild,Delhi,No
3,31,Female,98.0,Mild,Kolkata,No
4,65,Female,101.0,Mild,Mumbai,No


In [4]:
df.isnull().sum()

age           0
gender        0
fever        10
cough         0
city          0
has_covid     0
dtype: int64

In [6]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(df.drop(columns=['has_covid']), df['has_covid'], test_size=0.2)

In [7]:
X_train

,age,gender,fever,cough,city
59,6,Female,104.0,Mild,Kolkata
15,70,Male,103.0,Strong,Kolkata
11,65,Female,98.0,Mild,Mumbai
26,19,Female,100.0,Mild,Kolkata
98,5,Female,98.0,Strong,Mumbai
...,...,...,...,...,...
17,40,Female,98.0,Strong,Delhi
77,8,Female,101.0,Mild,Kolkata
7,20,Female,NaN,Strong,Mumbai
25,23,Male,NaN,Mild,Mumbai


# 1.Simple way of Transformer


In [10]:
# adding simple imputer to fever col

si = SimpleImputer()
X_train_fever = si.fit_transform(X_train[['fever']])


# also the test data
X_test_fever = si.fit_transform(X_test[['fever']])
# X_train_fever
X_train_fever.shape

(80, 1)

In [12]:
# Ordinalencoding -> cough
oe = OrdinalEncoder(categories=[['Mild', 'Strong']])
X_train_cough = oe.fit_transform(X_train[['cough']])

# also the test data
X_test_cough = oe.fit_transform(X_test[['cough']])
# X_train_cough
X_train_cough.shape

(80, 1)

In [15]:
# OneHotEncoding -> gender, city
ohe = OneHotEncoder(drop='first', sparse_output=False)
X_train_gender_city = ohe.fit_transform(X_train[['gender', 'city']])

# also the test data
X_test_gender_city = ohe.fit_transform(X_test[['gender', 'city']])
# X_train_gender_city
X_train_gender_city.shape

(80, 4)

In [18]:
# Extracting Age
X_train_age = X_train.drop(columns=['gender', 'fever', 'cough', 'city']).values

# also the test data
X_test_age = X_test.drop(columns=['gender', 'fever', 'cough', 'city']).values
# X_train_age
X_train_age.shape

(80, 1)

In [22]:
X_train_transformed = np.concatenate((X_train_age, X_train_fever, X_train_gender_city, X_train_cough), axis=1)
# also the test data

X_test_transformed = np.concatenate((X_test_age, X_test_fever, X_test_gender_city, X_test_cough) ,axis=1)
# X_train_transformed.shape
X_train_transformed

array([[  6.        , 104.        ,   0.        ,   0.        ,
          1.        ,   0.        ,   0.        ],
       [ 70.        , 103.        ,   1.        ,   0.        ,
          1.        ,   0.        ,   1.        ],
       [ 65.        ,  98.        ,   0.        ,   0.        ,
          0.        ,   1.        ,   0.        ],
       [ 19.        , 100.        ,   0.        ,   0.        ,
          1.        ,   0.        ,   0.        ],
       [  5.        ,  98.        ,   0.        ,   0.        ,
          0.        ,   1.        ,   1.        ],
       [ 49.        , 101.        ,   0.        ,   1.        ,
          0.        ,   0.        ,   0.        ],
       [ 65.        , 101.        ,   0.        ,   0.        ,
          0.        ,   1.        ,   0.        ],
       [ 50.        , 103.        ,   0.        ,   0.        ,
          1.        ,   0.        ,   0.        ],
       [ 71.        ,  98.        ,   0.        ,   0.        ,
          1.    

# Some Advanced way of Transformer

In [23]:
from sklearn.compose import ColumnTransformer

In [26]:
transformer = ColumnTransformer(transformers=[('tnf1', SimpleImputer(),['fever']), 
                                             ('tnf2', OrdinalEncoder(categories=[['Mild', 'Strong']]),['cough']),
                                             ('tnf3', OneHotEncoder(sparse_output=False, drop='first'), ['gender', 'city'])
                                            ],remainder='passthrough')
                        

In [35]:
# transformer.fit_transform(X_train)
transformer.fit_transform(X_train).shape

(80, 7)

In [32]:
transformer.transform(X_test)

array([[ 98.        ,   0.        ,   0.        ,   0.        ,
          0.        ,   0.        ,  64.        ],
       [100.91780822,   0.        ,   0.        ,   1.        ,
          0.        ,   0.        ,  75.        ],
       [100.        ,   1.        ,   0.        ,   0.        ,
          0.        ,   0.        ,  47.        ],
       [102.        ,   1.        ,   0.        ,   0.        ,
          1.        ,   0.        ,  82.        ],
       [100.        ,   1.        ,   0.        ,   0.        ,
          1.        ,   0.        ,  13.        ],
       [103.        ,   0.        ,   0.        ,   0.        ,
          0.        ,   0.        ,  16.        ],
       [ 98.        ,   1.        ,   0.        ,   0.        ,
          1.        ,   0.        ,  10.        ],
       [102.        ,   0.        ,   1.        ,   0.        ,
          1.        ,   0.        ,   5.        ],
       [ 98.        ,   1.        ,   0.        ,   0.        ,
          0.    

In [36]:
transformer.transform(X_test).shape

(20, 7)